# 04 — Dataset Construction for HOD-Mamba
**Reddit Opinion & Community Research — bridges all prior artifacts into PyTorch-ready batches**



In [1]:
import sys, importlib

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        get_ipython().system(f"pip install -q {pkg}")

for pkg, imp in [("pandas", None), ("pyarrow", None), ("numpy", None), ("torch", None)]:
    ensure(pkg, imp)

print("Packages OK.")


Packages OK.


In [2]:
import os, json, pickle
import numpy as np
import pandas as pd
import torch

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = "/content/drive/MyDrive/PhD/reddit_opinion_research"
else:
    BASE_DIR = os.environ.get("REDDIT_RESEARCH_BASE_DIR", "./reddit_opinion_research")

def load_contract(name):
    path = os.path.join(BASE_DIR, "processed_v2", name)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

raw_contract = load_contract("raw_enrichment_contract.json")
hypergraph_contract = load_contract("engineered/hypergraph_contract.json") if os.path.isfile(
    os.path.join(BASE_DIR, "processed_v2", "engineered", "hypergraph_contract.json")
) else load_contract("hypergraph_contract.json")
labeling_contract = load_contract("labeling_contract.json")
finetune_contract = load_contract("finetune_contract.json")

PROCESSED_V2_DIR = raw_contract["processed_v2_dir"]
DATASET_DIR = os.path.join(PROCESSED_V2_DIR, "dataset")
os.makedirs(DATASET_DIR, exist_ok=True)
print("DATASET_DIR:", DATASET_DIR)


Mounted at /content/drive
DATASET_DIR: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/dataset


## Load everything, and explicitly verify the embeddings-to-corpus alignment

In [3]:
text_corpus_orig = pd.read_parquet(raw_contract["paths"]["text_corpus"])  # order embeddings were computed in
text_corpus_labeled = pd.read_parquet(finetune_contract["paths"]["text_corpus_labeled_finetuned"])

assert len(text_corpus_orig) == len(text_corpus_labeled), "Row count mismatch between original and labeled corpus."
assert (text_corpus_orig["doc_id"].values == text_corpus_labeled["doc_id"].values).all(), \
    "Row order mismatch between text_corpus and the labeled version -- embeddings would be silently misaligned!"

embeddings = np.load(labeling_contract["paths"]["dziribert_embeddings"])
assert len(embeddings) == len(text_corpus_labeled), \
    f"Embedding count ({len(embeddings)}) doesn't match corpus size ({len(text_corpus_labeled)})."

print(f"Alignment verified: {len(embeddings):,} embeddings <-> {len(text_corpus_labeled):,} labeled documents, same row order.")

doc_id_to_row = {d: i for i, d in enumerate(text_corpus_labeled["doc_id"].values)}

hyperedges = pd.read_parquet(hypergraph_contract["paths"]["hyperedges"])
node_features = pd.read_parquet(raw_contract["paths"]["node_features"])
static_edges = pd.read_parquet(raw_contract["paths"]["static_edges"])
coparticipation_edges = pd.read_parquet(raw_contract["paths"]["coparticipation_edges"])

print(f"hyperedges: {len(hyperedges):,} | node_features: {len(node_features):,} | "
      f"static_edges: {len(static_edges):,} | coparticipation_edges: {len(coparticipation_edges):,}")


Alignment verified: 210,994 embeddings <-> 210,994 labeled documents, same row order.
hyperedges: 12,932 | node_features: 22,465 | static_edges: 157,953 | coparticipation_edges: 1,891,957


## Filter to multi-party hyperedges

In [4]:
n_before = len(hyperedges)
hyperedges = hyperedges[hyperedges["n_unique_authors"] >= 2].reset_index(drop=True)
print(f"Kept {len(hyperedges):,} / {n_before:,} hyperedges after filtering out single-author threads.")


Kept 9,200 / 12,932 hyperedges after filtering out single-author threads.


## Truncation policy: chronological prefix



In [5]:
MAX_SEQ_LEN = 180  # just above the 99th percentile (157.7) from notebook 02

# Structural proof-by-example: truncating at any length can't create a NEW dangling parent
_demo_docs = ["post_a", "comment_b", "comment_c", "comment_d"]
_demo_parents = [None, "post_a", "post_a", "comment_b"]
for L in range(1, len(_demo_docs) + 1):
    kept = set(_demo_docs[:L])
    newly_dangling = any(p is not None and p not in kept for p in _demo_parents[:L])
    assert not newly_dangling
print("Confirmed: chronological-prefix truncation introduces zero new dangling parents at any length.")
print("(The ~2.6% dangling parents from notebook 01b are pre-existing -- genuinely deleted/missing content --")
print(" not an artifact of this truncation choice.)")


Confirmed: chronological-prefix truncation introduces zero new dangling parents at any length.
(The ~2.6% dangling parents from notebook 01b are pre-existing -- genuinely deleted/missing content --
 not an artifact of this truncation choice.)


## Global vocabularies: authors and labels

In [6]:
author_to_idx = {a: i for i, a in enumerate(node_features["author_hash"].values)}

# Compute a real per-author average sentiment from the fine-tuned labels -- the old
# avg_sentiment column from notebook 01b was always an all-NA placeholder, never filled in.
sentiment_numeric_map = {"Negative": -1.0, "Neutral": 0.0, "Positive": 1.0}
sentiment_numeric = text_corpus_labeled["sentiment_label"].map(sentiment_numeric_map)
avg_sentiment_by_author = sentiment_numeric.groupby(text_corpus_labeled["author_hash"]).mean()
node_features["avg_sentiment_score"] = node_features["author_hash"].map(avg_sentiment_by_author)

# Dynamic column detection instead of hardcoding names -- avoids assuming a schema that
# doesn't match what notebook 01b actually produced (out_degree/in_degree, not "degree").
EXCLUDE_COLS = {"author_hash", "community_id", "avg_sentiment"}  # avg_sentiment is the stale placeholder above
node_feature_cols = [c for c in node_features.columns if c not in EXCLUDE_COLS]
print("Using node feature columns:", node_feature_cols)

def signed_log1p(x):
    # Handles both non-negative counts (degree, n_posts, n_comments) and columns that can be
    # negative (avg_score / karma, avg_sentiment_score in [-1,1]) without clipping sign away.
    return np.sign(x) * np.log1p(np.abs(x))

node_feat_matrix = node_features[node_feature_cols].fillna(0).values.astype("float64")
node_feat_matrix = signed_log1p(node_feat_matrix)
node_feat_matrix = (node_feat_matrix - node_feat_matrix.mean(axis=0)) / (node_feat_matrix.std(axis=0) + 1e-6)
node_feat_matrix = node_feat_matrix.astype("float32")
print(f"node_feat_matrix: {node_feat_matrix.shape} (columns: {node_feature_cols})")

sentiment_classes = sorted(text_corpus_labeled["sentiment_label"].dropna().unique())
emotion_classes = sorted(text_corpus_labeled["emotion_label"].dropna().unique())
sentiment_to_idx = {c: i for i, c in enumerate(sentiment_classes)}
emotion_to_idx = {c: i for i, c in enumerate(emotion_classes)}
print("Sentiment classes:", sentiment_to_idx)
print("Emotion classes:", emotion_to_idx)

doc_id_to_sentiment = text_corpus_labeled.set_index("doc_id")["sentiment_label"].map(sentiment_to_idx)
doc_id_to_emotion = text_corpus_labeled.set_index("doc_id")["emotion_label"].map(emotion_to_idx)


Using node feature columns: ['out_degree', 'in_degree', 'n_posts', 'n_comments', 'avg_score', 'avg_sentiment_score']
node_feat_matrix: (22465, 6) (columns: ['out_degree', 'in_degree', 'n_posts', 'n_comments', 'avg_score', 'avg_sentiment_score'])
Sentiment classes: {'Negative': 0, 'Neutral': 1, 'Positive': 2}
Emotion classes: {'Anger': 0, 'Joy': 1, 'Neutral': 2, 'Sadness': 3, 'Surprise': 4}


## Build per-hyperedge index arrays 

In [7]:
RELATION_TO_IDX = {None: 0, "reply_to_post": 1, "reply_to_comment": 2}

def build_item(row):
    doc_ids = row["doc_ids"][:MAX_SEQ_LEN]
    parent_doc_ids = row["parent_doc_ids"][:MAX_SEQ_LEN] if "parent_doc_ids" in row.index else [None] * len(doc_ids)
    reply_edge_types = row["reply_edge_types"][:MAX_SEQ_LEN] if "reply_edge_types" in row.index else [None] * len(doc_ids)
    authors = row["author_hashes"][:MAX_SEQ_LEN]
    timestamps = np.array(row["timestamps_unix"][:MAX_SEQ_LEN], dtype="float64")

    pos_lookup = {d: i for i, d in enumerate(doc_ids)}
    parent_idx = np.array([
        pos_lookup[p] if (p is not None and p in pos_lookup) else -1
        for p in parent_doc_ids
    ], dtype="int64")
    # relation_idx: which relation connects this position to its parent (0 if root or parent was
    # truncated/dangling -- consistent with parent_idx's own -1 fallback in that case).
    relation_idx = np.array([
        RELATION_TO_IDX.get(r, 0) if parent_idx[i] >= 0 else 0
        for i, r in enumerate(reply_edge_types)
    ], dtype="int64")

    # delta_t from parent (0 for root), log1p-scaled per the duration-tail finding from notebook 02
    parent_ts = np.where(parent_idx >= 0, timestamps[np.clip(parent_idx, 0, None)], timestamps)
    delta_t = np.log1p(np.clip(timestamps - parent_ts, a_min=0, a_max=None)).astype("float32")

    emb_idx = np.array([doc_id_to_row[d] for d in doc_ids], dtype="int64")
    author_idx = np.array([author_to_idx.get(a, -1) for a in authors], dtype="int64")
    sentiment_idx = np.array([doc_id_to_sentiment.get(d, -1) for d in doc_ids], dtype="int64")
    emotion_idx = np.array([doc_id_to_emotion.get(d, -1) for d in doc_ids], dtype="int64")

    return {
        "emb_idx": emb_idx, "parent_idx": parent_idx, "author_idx": author_idx,
        "relation_idx": relation_idx, "delta_t": delta_t,
        "sentiment_idx": sentiment_idx, "emotion_idx": emotion_idx,
        "length": len(doc_ids), "start_time": float(row["start_time_unix"]),
    }

items = [build_item(row) for _, row in hyperedges.iterrows()]
print(f"Built {len(items):,} hyperedge items.")
print("Example item 0:", {k: (v if not hasattr(v, 'shape') else v.shape) for k, v in items[0].items()})


Built 9,200 hyperedge items.
Example item 0: {'emb_idx': (21,), 'parent_idx': (21,), 'author_idx': (21,), 'relation_idx': (21,), 'delta_t': (21,), 'sentiment_idx': (21,), 'emotion_idx': (21,), 'length': 21, 'start_time': 1767226500.0}


## Temporal train/val/test split 

In [8]:
order = np.argsort([it["start_time"] for it in items])
n = len(order)
n_train, n_val = int(n * 0.70), int(n * 0.15)
train_idx = order[:n_train]
val_idx = order[n_train:n_train + n_val]
test_idx = order[n_train + n_val:]

splits = {"train": [items[i] for i in train_idx], "val": [items[i] for i in val_idx], "test": [items[i] for i in test_idx]}
for name, split_items in splits.items():
    print(f"{name}: {len(split_items):,} hyperedges, time range "
          f"{min(it['start_time'] for it in split_items):.0f} -> {max(it['start_time'] for it in split_items):.0f}")


train: 6,440 hyperedges, time range 1767225715 -> 1777313158
val: 1,380 hyperedges, time range 1777315389 -> 1780096638
test: 1,380 hyperedges, time range 1780098412 -> 1782862481


## PyTorch Dataset + collate_fn

In [9]:
class HyperedgeDataset(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def make_collate_fn(embeddings, node_feat_matrix):
    emb_dim = embeddings.shape[1]
    n_node_feats = node_feat_matrix.shape[1]

    def collate(batch):
        B = len(batch)
        L = max(it["length"] for it in batch)
        emb_batch = np.zeros((B, L, emb_dim), dtype="float32")
        node_feat_batch = np.zeros((B, L, n_node_feats), dtype="float32")
        parent_batch = np.full((B, L), -1, dtype="int64")
        relation_batch = np.zeros((B, L), dtype="int64")  # 0 = none/root, matches RELATION_TO_IDX
        author_batch = np.full((B, L), -1, dtype="int64")
        delta_t_batch = np.zeros((B, L), dtype="float32")
        sentiment_batch = np.full((B, L), -1, dtype="int64")
        emotion_batch = np.full((B, L), -1, dtype="int64")
        mask = np.zeros((B, L), dtype="bool")

        for i, it in enumerate(batch):
            n = it["length"]
            emb_batch[i, :n] = embeddings[it["emb_idx"]]
            valid_author = it["author_idx"] >= 0
            node_feat_batch[i, :n][valid_author] = node_feat_matrix[it["author_idx"][valid_author]]
            parent_batch[i, :n] = it["parent_idx"]
            relation_batch[i, :n] = it["relation_idx"]
            author_batch[i, :n] = it["author_idx"]
            delta_t_batch[i, :n] = it["delta_t"]
            sentiment_batch[i, :n] = it["sentiment_idx"]
            emotion_batch[i, :n] = it["emotion_idx"]
            mask[i, :n] = True

        return {
            "embeddings": torch.from_numpy(emb_batch),
            "node_features": torch.from_numpy(node_feat_batch),
            "parent_idx": torch.from_numpy(parent_batch),
            "relation_idx": torch.from_numpy(relation_batch),
            "author_idx": torch.from_numpy(author_batch),
            "delta_t": torch.from_numpy(delta_t_batch),
            "sentiment_idx": torch.from_numpy(sentiment_batch),
            "emotion_idx": torch.from_numpy(emotion_batch),
            "mask": torch.from_numpy(mask),
        }

    return collate

collate_fn = make_collate_fn(embeddings, node_feat_matrix)
train_loader = torch.utils.data.DataLoader(
    HyperedgeDataset(splits["train"]), batch_size=32, shuffle=True, collate_fn=collate_fn
)

batch = next(iter(train_loader))
for k, v in batch.items():
    print(f"{k}: {tuple(v.shape)} {v.dtype}")

row_lengths = batch["mask"].sum(dim=1)
assert (row_lengths > 0).all(), "Every hyperedge in the batch should have at least one valid document."
assert (row_lengths <= batch["mask"].shape[1]).all()
assert batch["parent_idx"][batch["mask"]].min().item() >= -1
print("\nBatch constructed successfully -- per-row lengths:", row_lengths.tolist())


embeddings: (32, 129, 768) torch.float32
node_features: (32, 129, 6) torch.float32
parent_idx: (32, 129) torch.int64
relation_idx: (32, 129) torch.int64
author_idx: (32, 129) torch.int64
delta_t: (32, 129) torch.float32
sentiment_idx: (32, 129) torch.int64
emotion_idx: (32, 129) torch.int64
mask: (32, 129) torch.bool

Batch constructed successfully -- per-row lengths: [4, 8, 3, 2, 2, 4, 19, 24, 35, 6, 2, 9, 27, 33, 59, 129, 16, 28, 127, 17, 15, 10, 12, 11, 11, 3, 4, 8, 2, 12, 18, 12]


## Global static author-author graph 

In [10]:
static_edges_idx = static_edges.copy()
static_edges_idx["source_idx"] = static_edges_idx["source_author"].map(author_to_idx)
static_edges_idx["target_idx"] = static_edges_idx["target_author"].map(author_to_idx)
static_edges_idx = static_edges_idx.dropna(subset=["source_idx", "target_idx"])
static_edges_idx["relation"] = static_edges_idx["edge_type"]

coparticipation_static = (
    coparticipation_edges.groupby(["source_author", "target_author"]).size().reset_index(name="weight")
)
coparticipation_static["source_idx"] = coparticipation_static["source_author"].map(author_to_idx)
coparticipation_static["target_idx"] = coparticipation_static["target_author"].map(author_to_idx)
coparticipation_static = coparticipation_static.dropna(subset=["source_idx", "target_idx"])
coparticipation_static["relation"] = "co_participation"

combined_graph = pd.concat([
    static_edges_idx[["source_idx", "target_idx", "weight", "relation"]],
    coparticipation_static[["source_idx", "target_idx", "weight", "relation"]],
], ignore_index=True)
combined_graph["source_idx"] = combined_graph["source_idx"].astype(int)
combined_graph["target_idx"] = combined_graph["target_idx"].astype(int)

print(f"Combined static graph: {len(combined_graph):,} edges across relations:")
print(combined_graph["relation"].value_counts())

graph_path = os.path.join(DATASET_DIR, "static_author_graph.parquet")
combined_graph.to_parquet(graph_path, engine="pyarrow", index=False)
print("Saved:", graph_path)


Combined static graph: 1,807,555 edges across relations:
relation
co_participation    1649602
reply_to_post         79394
reply_to_comment      78559
Name: count, dtype: int64
Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/dataset/static_author_graph.parquet


## Save the splits, vocabularies, and dataset contract

In [11]:
for name, split_items in splits.items():
    with open(os.path.join(DATASET_DIR, f"{name}_items.pkl"), "wb") as f:
        pickle.dump(split_items, f)

np.save(os.path.join(DATASET_DIR, "node_feat_matrix.npy"), node_feat_matrix)

with open(os.path.join(DATASET_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump({
        "author_to_idx": author_to_idx,
        "sentiment_to_idx": sentiment_to_idx,
        "emotion_to_idx": emotion_to_idx,
        "relation_to_idx": {str(k): v for k, v in RELATION_TO_IDX.items()},
        "node_feature_cols": node_feature_cols,
        "max_seq_len": MAX_SEQ_LEN,
    }, f, ensure_ascii=False)

dataset_contract = {
    "dataset_dir": DATASET_DIR,
    "embeddings_path": labeling_contract["paths"]["dziribert_embeddings"],
    "embedding_dim": int(embeddings.shape[1]),
    "n_authors": len(author_to_idx),
    "n_node_features": len(node_feature_cols),
    "max_seq_len": MAX_SEQ_LEN,
    "split_sizes": {k: len(v) for k, v in splits.items()},
    "static_graph_path": graph_path,
    "vocab_path": os.path.join(DATASET_DIR, "vocab.json"),
}
with open(os.path.join(DATASET_DIR, "dataset_contract.json"), "w", encoding="utf-8") as f:
    json.dump(dataset_contract, f, indent=2, ensure_ascii=False)

print(json.dumps(dataset_contract, indent=2, ensure_ascii=False))


{
  "dataset_dir": "/content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/dataset",
  "embeddings_path": "/content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/dziribert_embeddings.npy",
  "embedding_dim": 768,
  "n_authors": 22465,
  "n_node_features": 6,
  "max_seq_len": 180,
  "split_sizes": {
    "train": 6440,
    "val": 1380,
    "test": 1380
  },
  "static_graph_path": "/content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/dataset/static_author_graph.parquet",
  "vocab_path": "/content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/dataset/vocab.json"
}
